# Wording Impact
How does the wording of a listing impact revenue?

In [ ]:
import pandas as pd

berlin = pd.read_csv('https://raw.githubusercontent.com/HereOnGithub/airbnb-dashboard-2026/main/data/zip_berlin_listings.csv')
munich = pd.read_csv('https://raw.githubusercontent.com/HereOnGithub/airbnb-dashboard-2026/main/data/zip_munich_listings.csv')

In [ ]:
def analyze(df, city):
    df = df.copy()
    df['price'] = df['price'].replace('[\€\$,]', '', regex=True)
    df['price'] = pd.to_numeric(df['price'], errors='coerce')
    df = df[df['price'] > 0]
    df['desc_wc'] = df['description'].fillna('').apply(lambda x: len(str(x).split()))
    df['revenue'] = pd.to_numeric(df['estimated_revenue_l365d'], errors='coerce').fillna(0)
    df['rating'] = pd.to_numeric(df['review_scores_rating'], errors='coerce')

    bins = [0, 10, 50, 100, 200, 5000]
    labels = ['Minimal (0-10)', 'Brief (11-50)', 'Moderate (51-100)', 'Detailed (101-200)', 'Max Detail (200+)']
    df['bin'] = pd.cut(df['desc_wc'], bins=bins, labels=labels)

    result = df.groupby('bin', observed=True).agg(
        listings=('revenue', 'count'),
        avg_revenue=('revenue', 'mean'),
        avg_rating=('rating', lambda x: x.dropna().mean())
    ).round(2)

    print(f'--- {city} ---')
    print(result)
    print()

analyze(berlin, 'Berlin')
analyze(munich, 'Munich')